# Dataset Comparison: Benchmark vs Comp Period Files

This notebook compares the `benchmark_weights_carbon_intensity_{period}.xlsx` files with their corresponding `dataset_comp_{period}.xlsx` files across all time periods.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Define Periods and Load Data

In [ ]:
# Define periods
periods = ['1221', '0322', '0622', '0922', '1222']

# Data directory
data_dir = Path('data/datasets')

# Dictionary to store loaded data
benchmark_data = {}
comp_data = {}

# Load all datasets
for period in periods:
    # Load benchmark file
    benchmark_file = data_dir / f'benchmark_weights_carbon_intensity_{period}.xlsx'
    if benchmark_file.exists():
        benchmark_data[period] = pd.read_excel(benchmark_file)
        print(f"Loaded benchmark data for {period}: {len(benchmark_data[period])} rows")
    else:
        print(f"Warning: Benchmark file not found for {period}")
    
    # Load comp file
    comp_file = data_dir / f'dataset_comp_{period}.xlsx'
    if comp_file.exists():
        comp_data[period] = pd.read_excel(comp_file)
        print(f"Loaded comp data for {period}: {len(comp_data[period])} rows")
    else:
        print(f"Warning: Comp file not found for {period}")
    print()

## 2. Compare Number of Companies Per Period

In [ ]:
# Create comparison dataframe
comparison_summary = pd.DataFrame({
    'Period': periods,
    'Benchmark Companies': [len(benchmark_data.get(p, [])) for p in periods],
    'Comp Dataset Companies': [len(comp_data.get(p, [])) for p in periods]
})

comparison_summary['Match'] = comparison_summary['Benchmark Companies'] == comparison_summary['Comp Dataset Companies']

print("\nCompany Count Comparison:")
print(comparison_summary)
print(f"\nAll periods match: {comparison_summary['Match'].all()}")

## 3. Compare Common Columns Between Files

In [ ]:
# Common columns between benchmark and comp datasets
common_cols = ['SYMBOL', 'NAME', 'GICS Sector', 'Carbon Intensity', 'weight_in_sector']

print("Columns in benchmark files:", benchmark_data[periods[0]].columns.tolist())
print("\nColumns in comp files:", comp_data[periods[0]].columns.tolist())
print("\nCommon columns to compare:", common_cols)

## 4. Detailed Comparison for Each Period

In [ ]:
def compare_datasets(period, benchmark_df, comp_df):
    """
    Compare benchmark and comp datasets for a specific period
    """
    print(f"\n{'='*80}")
    print(f"Period: {period}")
    print(f"{'='*80}")
    
    # 1. Check if same companies (by SYMBOL)
    benchmark_symbols = set(benchmark_df['SYMBOL'])
    comp_symbols = set(comp_df['SYMBOL'])
    
    only_in_benchmark = benchmark_symbols - comp_symbols
    only_in_comp = comp_symbols - benchmark_symbols
    common_symbols = benchmark_symbols & comp_symbols
    
    print(f"\n1. SYMBOL Comparison:")
    print(f"   - Common symbols: {len(common_symbols)}")
    print(f"   - Only in benchmark: {len(only_in_benchmark)}")
    if only_in_benchmark:
        print(f"     {only_in_benchmark}")
    print(f"   - Only in comp: {len(only_in_comp)}")
    if only_in_comp:
        print(f"     {only_in_comp}")
    
    # 2. Compare common columns for matching symbols
    if len(common_symbols) > 0:
        # Merge on SYMBOL
        merged = pd.merge(
            benchmark_df[common_cols],
            comp_df[common_cols],
            on='SYMBOL',
            suffixes=('_benchmark', '_comp')
        )
        
        print(f"\n2. Column-by-Column Comparison:")
        
        # Compare GICS Sector
        sector_mismatch = merged[merged['GICS Sector_benchmark'] != merged['GICS Sector_comp']]
        print(f"   - GICS Sector mismatches: {len(sector_mismatch)}")
        if len(sector_mismatch) > 0:
            print(sector_mismatch[['SYMBOL', 'GICS Sector_benchmark', 'GICS Sector_comp']])
        
        # Compare Carbon Intensity (numerical - allow small differences)
        ci_diff = abs(merged['Carbon Intensity_benchmark'] - merged['Carbon Intensity_comp'])
        ci_mismatch = merged[ci_diff > 0.0001]
        print(f"\n   - Carbon Intensity mismatches (diff > 0.0001): {len(ci_mismatch)}")
        if len(ci_mismatch) > 0:
            print(ci_mismatch[['SYMBOL', 'Carbon Intensity_benchmark', 'Carbon Intensity_comp']])
            print(f"     Max difference: {ci_diff.max():.6f}")
            print(f"     Mean difference: {ci_diff.mean():.6f}")
        
        # Compare weights
        weight_diff = abs(merged['weight_in_sector_benchmark'] - merged['weight_in_sector_comp'])
        weight_mismatch = merged[weight_diff > 0.0001]
        print(f"\n   - Weight in sector mismatches (diff > 0.0001): {len(weight_mismatch)}")
        if len(weight_mismatch) > 0:
            print(weight_mismatch[['SYMBOL', 'weight_in_sector_benchmark', 'weight_in_sector_comp']])
            print(f"     Max difference: {weight_diff.max():.6f}")
            print(f"     Mean difference: {weight_diff.mean():.6f}")
        
        # 3. Summary statistics comparison
        print(f"\n3. Summary Statistics:")
        stats_comparison = pd.DataFrame({
            'Metric': ['Mean Carbon Intensity', 'Std Carbon Intensity', 'Mean Weight', 'Sum of Weights'],
            'Benchmark': [
                benchmark_df['Carbon Intensity'].mean(),
                benchmark_df['Carbon Intensity'].std(),
                benchmark_df['weight_in_sector'].mean(),
                benchmark_df['weight_in_sector'].sum()
            ],
            'Comp': [
                comp_df['Carbon Intensity'].mean(),
                comp_df['Carbon Intensity'].std(),
                comp_df['weight_in_sector'].mean(),
                comp_df['weight_in_sector'].sum()
            ]
        })
        stats_comparison['Difference'] = stats_comparison['Benchmark'] - stats_comparison['Comp']
        print(stats_comparison)
        
        return merged
    
    return None

# Run comparison for all periods
merged_data = {}
for period in periods:
    if period in benchmark_data and period in comp_data:
        merged_data[period] = compare_datasets(period, benchmark_data[period], comp_data[period])

## 5. Sector-Level Comparison

In [ ]:
# Compare number of companies per sector
def compare_sector_composition(period, benchmark_df, comp_df):
    """
    Compare sector composition between benchmark and comp datasets
    """
    print(f"\n{'='*80}")
    print(f"Sector Composition for Period: {period}")
    print(f"{'='*80}")
    
    benchmark_sectors = benchmark_df['GICS Sector'].value_counts().sort_index()
    comp_sectors = comp_df['GICS Sector'].value_counts().sort_index()
    
    sector_comparison = pd.DataFrame({
        'Benchmark': benchmark_sectors,
        'Comp': comp_sectors
    }).fillna(0).astype(int)
    
    sector_comparison['Difference'] = sector_comparison['Benchmark'] - sector_comparison['Comp']
    sector_comparison['Match'] = sector_comparison['Difference'] == 0
    
    print(sector_comparison)
    print(f"\nAll sectors match: {sector_comparison['Match'].all()}")
    
    return sector_comparison

# Run sector comparison for all periods
sector_comparisons = {}
for period in periods:
    if period in benchmark_data and period in comp_data:
        sector_comparisons[period] = compare_sector_composition(period, benchmark_data[period], comp_data[period])

## 6. Visualize Differences Across Periods

In [ ]:
# Plot carbon intensity comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, period in enumerate(periods):
    if period in merged_data and merged_data[period] is not None:
        merged = merged_data[period]
        
        ax = axes[idx]
        ax.scatter(
            merged['Carbon Intensity_benchmark'],
            merged['Carbon Intensity_comp'],
            alpha=0.5
        )
        
        # Add diagonal line (y=x)
        max_val = max(
            merged['Carbon Intensity_benchmark'].max(),
            merged['Carbon Intensity_comp'].max()
        )
        ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='y=x')
        
        ax.set_xlabel('Benchmark Carbon Intensity')
        ax.set_ylabel('Comp Carbon Intensity')
        ax.set_title(f'Period {period}')
        ax.legend()
        ax.grid(True, alpha=0.3)

# Remove extra subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.savefig('figures/carbon_intensity_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved figure to: figures/carbon_intensity_comparison.png")

In [ ]:
# Plot weight comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, period in enumerate(periods):
    if period in merged_data and merged_data[period] is not None:
        merged = merged_data[period]
        
        ax = axes[idx]
        ax.scatter(
            merged['weight_in_sector_benchmark'],
            merged['weight_in_sector_comp'],
            alpha=0.5
        )
        
        # Add diagonal line (y=x)
        max_val = max(
            merged['weight_in_sector_benchmark'].max(),
            merged['weight_in_sector_comp'].max()
        )
        ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='y=x')
        
        ax.set_xlabel('Benchmark Weight in Sector')
        ax.set_ylabel('Comp Weight in Sector')
        ax.set_title(f'Period {period}')
        ax.legend()
        ax.grid(True, alpha=0.3)

# Remove extra subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.savefig('figures/weight_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved figure to: figures/weight_comparison.png")

## 7. Additional Columns in Comp Dataset

In [ ]:
# Show what additional information is in the comp dataset
print("Additional columns in comp dataset:")
comp_cols = set(comp_data[periods[0]].columns)
benchmark_cols = set(benchmark_data[periods[0]].columns)
additional_cols = comp_cols - benchmark_cols

print(f"\nBenchmark columns: {sorted(benchmark_cols)}")
print(f"\nAdditional columns in comp dataset: {sorted(additional_cols)}")

# Show sample data from comp dataset
print("\nSample data from comp dataset (first 5 rows):")
print(comp_data[periods[0]].head())

## 8. Summary Report

In [ ]:
print("="*80)
print("SUMMARY REPORT")
print("="*80)

print("\n1. Files loaded:")
print(f"   - Benchmark files: {len(benchmark_data)}")
print(f"   - Comp files: {len(comp_data)}")

print("\n2. Key findings:")
print("   - The benchmark files contain a subset of columns from the comp files")
print("   - Both files should contain the same companies (same SYMBOLs)")
print("   - The benchmark files are simplified versions focusing on:")
print("     * SYMBOL, NAME, GICS Sector")
print("     * Carbon Intensity")
print("     * weight_in_sector")
print("   - The comp files contain additional details:")
print("     * Price and share data")
print("     * Individual scope emissions (1, 2, 3)")
print("     * Revenue data")
print("     * Imputation flags")
print("     * Ranking information")

print("\n3. Use cases:")
print("   - Benchmark files: Quick reference for portfolio optimization")
print("   - Comp files: Detailed analysis and auditing")

print("\n" + "="*80)